# 🧠 KORA-0.01 — Notebook 2 : le cerveau

On entraîne le modèle qui décide si un message est une arnaque, et de quel type.

**Le plan**
1. **Une référence simple (baseline)** : une méthode classique, sans réseau de neurones. Elle sert d'étalon. Si notre « grosse IA » ne fait pas mieux, elle ne sert à rien.
2. **Le vrai modèle KORA** : on part d'**AfroXLMR-large-76L**, un modèle de langage qui a lu des textes dans **76 langues africaines, dont le fon et le français** (licence MIT, utilisable commercialement). On l'**affine** (*fine-tuning*) sur nos messages, avec le GPU.
3. **L'examen** : on mesure sur le test (arnaques jamais vues), sur le fon traduit par des humains et sur les vrais messages récoltés.
4. **La sauvegarde** : le modèle part sur le Drive (et sur Hugging Face si tu veux).

⚙️ **Avant de commencer** : Exécution → Modifier le type d'exécution → **T4 GPU**.
⏱ Durée : ~15 à 25 minutes sur GPU T4.

In [ ]:
# Installation des bibliothèques (Colab a déjà PyTorch)
import sys
if "google.colab" in sys.modules:
    !pip -q install -U transformers accelerate

In [ ]:
# ▶ Connexion à Google Drive et chemins du projet
# En Colab : le dossier KORA doit être dans "Mon Drive" (MyDrive/KORA).
# En local : le notebook utilise le dossier parent.
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/KORA"
else:
    BASE = os.path.abspath("..")
sys.path.append(BASE)
for d in ["data", "modeles", "resultats"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)
print("Dossier du projet :", BASE)
print("Contenu :", sorted(os.listdir(BASE)))
from kora_lib import CATEGORIES, NOMS, normaliser, expliquer
ID = {c: i for i, c in enumerate(CATEGORIES)}      # "demande_code" -> 1
print(ID)

## 1. Charger les données

On recharge ce que le notebook 1 a fabriqué, et on applique `normaliser()` (minuscules, sans tons, liens → `lien`, numéros → `numtel`).

**Pourquoi remplacer les numéros et les liens ?** Pour que le modèle apprenne la **forme** d'une arnaque (« envoie le code au *numéro* »), pas un numéro précis qu'un arnaqueur changera demain.

In [ ]:
import pandas as pd

df = pd.read_csv(f"{BASE}/data/kora_fr.csv")

INCLURE_FON_BROUILLON = True     # brouillons fon NLLB (notebook 1) ajoutés à l'entraînement
chemin_fon = f"{BASE}/data/kora_fon_brouillon.csv"
if INCLURE_FON_BROUILLON and os.path.exists(chemin_fon):
    fon = pd.read_csv(chemin_fon)
    fon["split"] = "train"
    df = pd.concat([df, fon[df.columns.intersection(fon.columns)]], ignore_index=True)
    print(f"+ {len(fon)} brouillons fon ajoutés à l'entraînement")

df["x"] = df["texte"].map(normaliser)
df["y"] = df["categorie"].map(ID)
train, val, test = (df[df["split"] == s].reset_index(drop=True) for s in ["train", "val", "test"])

# Tests bonus : fon traduit par des humains, vrais messages du terrain
tests_bonus = {}
p = f"{BASE}/data/a_traduire_en_fon.csv"
if os.path.exists(p):
    t = pd.read_csv(p).fillna("")
    t = t[t["texte_fon"].str.strip() != ""]
    # sécurité anti-fuite : on ne garde que les phrases qui sont bien dans le TEST actuel
    hors_test = ~t["texte_fr"].isin(set(test["texte"]))
    if hors_test.any():
        print(f"⚠️ {int(hors_test.sum())} traductions fon ignorées : leur phrase n'est plus dans le test "
              "(le notebook 1 a été relancé avec d'autres modèles). Re-génère a_traduire_en_fon.csv.")
        t = t[~hors_test]
    if len(t):
        tests_bonus["fon_humain"] = pd.DataFrame({"texte": t["texte_fon"], "categorie": t["categorie"]})
p = f"{BASE}/data/vrais_messages.csv"
if os.path.exists(p):
    tests_bonus["vrais_messages"] = pd.read_csv(p)[["texte", "categorie"]]
for nom, t in tests_bonus.items():
    t["x"] = t["texte"].map(normaliser)
    t["y"] = t["categorie"].map(ID)

print(f"train {len(train)} · val {len(val)} · test {len(test)} · tests bonus : {[f'{k} ({len(v)})' for k, v in tests_bonus.items()] or 'aucun pour l’instant'}")
train[["categorie", "x"]].head()

## 2. Comment on note le modèle

On ne regarde pas qu'un seul chiffre. Pour un anti-arnaque, deux erreurs n'ont pas le même coût :

| Erreur | Conséquence |
|---|---|
| **Arnaque ratée** (on dit « sûr » alors que c'est une arnaque) | La personne perd son argent 💸 |
| **Fausse alerte** (on dit « arnaque » alors que c'est normal) | L'utilisateur est agacé et finit par ignorer KORA |

Donc on mesure :
- **Taux de détection** : parmi les vraies arnaques, combien KORA en attrape (on vise le plus haut possible).
- **Taux de fausses alertes** : parmi les messages normaux, combien KORA accuse à tort (on vise le plus bas possible).
- **F1 macro** : la note globale sur les 5 catégories (0 = nul, 1 = parfait), chaque catégorie compte autant.

Ce sont exactement les indicateurs demandés par l'équipe finance (« précision des alertes », « taux de faux positifs »).

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

def evaluer(nom, y_vrai, y_pred, afficher=True):
    y_vrai, y_pred = np.asarray(y_vrai), np.asarray(y_pred)
    arnaque_vraie, arnaque_predite = y_vrai != ID["legitime"], y_pred != ID["legitime"]
    res = {
        "jeu": nom,
        "n": int(len(y_vrai)),
        "exactitude": round(accuracy_score(y_vrai, y_pred), 4),
        "f1_macro": round(f1_score(y_vrai, y_pred, average="macro", labels=list(range(len(CATEGORIES))), zero_division=0), 4),
        "taux_detection": round(float(arnaque_predite[arnaque_vraie].mean()) if arnaque_vraie.any() else float("nan"), 4),
        "taux_fausses_alertes": round(float(arnaque_predite[~arnaque_vraie].mean()) if (~arnaque_vraie).any() else float("nan"), 4),
    }
    if afficher:
        print(f"── {nom} ──  exactitude {res['exactitude']:.1%} · F1 macro {res['f1_macro']:.3f} · "
              f"détection {res['taux_detection']:.1%} · fausses alertes {res['taux_fausses_alertes']:.1%}")
    return res

RESULTATS = []

## 3. La référence simple (baseline) : TF-IDF + régression logistique

**Comment elle marche, en une phrase :** elle découpe chaque message en petits morceaux de lettres (« cod », « ode », « envoi »…), compte lesquels apparaissent, et apprend quels morceaux sont typiques de chaque catégorie.

C'est rapide (quelques secondes, sans GPU), facile à expliquer, et ça donne un **point de comparaison**. On la garde aussi comme **roue de secours** pour la démo si le GPU n'est pas disponible.

In [ ]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

baseline = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=3000, C=5, class_weight="balanced"),
)
baseline.fit(train["x"], train["y"])

RESULTATS.append({"modele": "baseline", **evaluer("val", val["y"], baseline.predict(val["x"]))})
RESULTATS.append({"modele": "baseline", **evaluer("test", test["y"], baseline.predict(test["x"]))})
for nom, t in tests_bonus.items():
    RESULTATS.append({"modele": "baseline", **evaluer(nom, t["y"], baseline.predict(t["x"]))})

joblib.dump(baseline, f"{BASE}/modeles/baseline.joblib")
print("💾 baseline sauvegardée")

### 3 bis. La validation croisée : un score qui ne dépend pas de la chance

Un seul découpage train / test, c'est **un seul tirage au sort** : selon les phrases à trous qui tombent dans le test, le score peut être très bon ou très mauvais. Pendant la préparation, la même baseline est passée de **70 %** à **99,6 %** de détection juste en changeant le tirage.

👉 On refait donc l'examen **5 fois** (5 « plis ») : à chaque fois, un groupe différent de phrases à trous sert de test, jamais vu à l'entraînement. On présente **la moyenne et l'écart** (± écart-type). C'est ce chiffre qu'il faut donner au jury : il est beaucoup plus fiable qu'un seul test.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import GroupKFold

tout = df.reset_index(drop=True)
plis = []
for k, (i_tr, i_te) in enumerate(GroupKFold(n_splits=5).split(tout, tout["y"], groups=tout["modele_id"])):
    m = clone(baseline).fit(tout["x"][i_tr], tout["y"][i_tr])
    plis.append(evaluer(f"pli {k+1}", tout["y"][i_te], m.predict(tout["x"][i_te])))
cv = pd.DataFrame(plis)
resume = {c: (np.nanmean(cv[c]), np.nanstd(cv[c])) for c in ["f1_macro", "taux_detection", "taux_fausses_alertes"]}
print("\n📊 Baseline, validation croisée 5 plis (moyenne ± écart) :")
for c, (moy, ec) in resume.items():
    print(f"   {c:22s} {moy:.1%} ± {ec:.1%}")
RESULTATS.append({"modele": "baseline", "jeu": "validation croisée 5 plis (moyenne)", "n": int(len(tout)),
                  "f1_macro": round(resume["f1_macro"][0], 4), "taux_detection": round(resume["taux_detection"][0], 4),
                  "taux_fausses_alertes": round(resume["taux_fausses_alertes"][0], 4)})

## 4. Le vrai modèle : fine-tuning d'AfroXLMR sur le GPU

**L'idée du fine-tuning, avec une image :** AfroXLMR a déjà « lu » énormément de textes africains, il connaît les langues, comme un étudiant qui sait lire le français et le fon. Mais il ne sait rien des arnaques. On lui ajoute une petite « tête » de décision (5 sorties, une par catégorie) et on lui fait réviser **nos** exemples plusieurs fois (les *epochs*). À chaque erreur, il ajuste un peu ses paramètres.

**Les réglages, et pourquoi :**
| Réglage | Valeur | Pourquoi |
|---|---|---|
| `TAUX_APPRENTISSAGE` | 1e-5 | petit pas : le modèle est gros, de grands pas lui feraient « oublier » ce qu'il sait |
| `EPOCHS` | 4 | 4 passages sur les données suffisent ; plus = risque d'apprendre par cœur |
| `LONGUEUR_MAX` | 128 | un SMS tient largement en 128 *tokens* (morceaux de mots) |
| `fp16` | oui | calcul en demi-précision : 2× plus rapide sur le T4, moins de mémoire |

À la fin de chaque epoch, on évalue sur **val** et on garde **la meilleure version** (pas forcément la dernière).

🧯 **Si la mémoire GPU déborde** (`CUDA out of memory`) : mets `TAILLE_LOT = 8`. **Si le score reste bloqué très bas** (le modèle prédit toujours la même catégorie, ça arrive parfois aux gros modèles), relance avec `GRAINE = 1`, ou passe `MODELE_BASE = "xlm-roberta-base"` (plus petit et plus stable, mais sans le fon dans son apprentissage).

In [ ]:
MODELE_BASE = "Davlan/afro-xlmr-large-76L"   # alternatives (voir MODELES.md) : "Davlan/afro-xlmr-base", "xlm-roberta-base"
TAUX_APPRENTISSAGE = 1e-5
EPOCHS = 4
TAILLE_LOT = 16
LONGUEUR_MAX = 128
GRAINE = 42

try:
    import torch
    GPU = torch.cuda.is_available()
except ImportError:
    GPU = False
ENTRAINER = GPU
print("GPU :", torch.cuda.get_device_name(0) if GPU else "❌ aucun (active le GPU T4 pour entraîner KORA)")

In [ ]:
if ENTRAINER:
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                              Trainer, DataCollatorWithPadding, set_seed)
    set_seed(GRAINE)
    tokenizer = AutoTokenizer.from_pretrained(MODELE_BASE)

    # Le tokenizer découpe le texte en morceaux et les transforme en numéros
    exemple = tokenizer.tokenize(train["x"][0])
    print("Exemple de découpage :", exemple[:20])

    class Messages(torch.utils.data.Dataset):
        def __init__(self, textes, etiquettes=None):
            self.enc = tokenizer(list(textes), truncation=True, max_length=LONGUEUR_MAX)
            self.y = None if etiquettes is None else list(etiquettes)
        def __len__(self):
            return len(self.enc["input_ids"])
        def __getitem__(self, i):
            item = {k: v[i] for k, v in self.enc.items()}
            if self.y is not None:
                item["labels"] = int(self.y[i])
            return item

    ds_train, ds_val = Messages(train["x"], train["y"]), Messages(val["x"], val["y"])

    modele = AutoModelForSequenceClassification.from_pretrained(
        MODELE_BASE, num_labels=len(CATEGORIES),
        id2label={i: c for i, c in enumerate(CATEGORIES)}, label2id=ID)

    def metriques(pred):
        y_pred = pred.predictions.argmax(-1)
        return {"f1_macro": f1_score(pred.label_ids, y_pred, average="macro"),
                "exactitude": accuracy_score(pred.label_ids, y_pred)}

    args = TrainingArguments(
        output_dir="/content/kora_entrainement" if IN_COLAB else "./kora_entrainement",
        learning_rate=TAUX_APPRENTISSAGE, num_train_epochs=EPOCHS,
        per_device_train_batch_size=TAILLE_LOT, per_device_eval_batch_size=64,
        warmup_ratio=0.1, weight_decay=0.01, fp16=True,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="f1_macro",
        logging_steps=20, report_to="none", seed=GRAINE,
    )
    trainer = Trainer(model=modele, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                      data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=metriques)
    trainer.train()
else:
    print("⏭ Entraînement sauté (pas de GPU). La baseline reste disponible.")

In [ ]:
# Courbe d'apprentissage : la perte (l'erreur) doit descendre
if ENTRAINER:
    import matplotlib.pyplot as plt
    hist = pd.DataFrame(trainer.state.log_history)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    h = hist.dropna(subset=["loss"])
    ax[0].plot(h["step"], h["loss"]); ax[0].set_title("Perte à l'entraînement (doit descendre)"); ax[0].set_xlabel("étape")
    h = hist.dropna(subset=["eval_f1_macro"])
    ax[1].plot(h["epoch"], h["eval_f1_macro"], marker="o"); ax[1].set_ylim(0, 1.02)
    ax[1].set_title("F1 macro sur la validation (doit monter)"); ax[1].set_xlabel("epoch")
    plt.tight_layout(); plt.savefig(f"{BASE}/resultats/courbes_apprentissage.png", dpi=150); plt.show()
    if h["eval_f1_macro"].max() < 0.4:
        print("⚠️ Le modèle n'a pas appris (F1 < 0.4). Relance avec GRAINE = 1 ou MODELE_BASE = 'xlm-roberta-base'.")

## 5. L'examen final

On évalue **une seule fois** sur le test. C'est le chiffre honnête à présenter au jury.

In [ ]:
def probas_kora(textes):
    """Probabilités de chaque catégorie pour une liste de textes (déjà normalisés)."""
    if ENTRAINER:
        sortie = trainer.predict(Messages(textes))
        return torch.softmax(torch.tensor(sortie.predictions, dtype=torch.float32), dim=-1).numpy()
    return baseline.predict_proba(list(textes))

NOM_KORA = "KORA (AfroXLMR affiné)" if ENTRAINER else "baseline"
predictions = {}
for nom, t in [("val", val), ("test", test), *tests_bonus.items()]:
    p = probas_kora(t["x"])
    predictions[nom] = p
    if ENTRAINER:
        RESULTATS.append({"modele": NOM_KORA, **evaluer(nom, t["y"], p.argmax(1))})

print()
print(classification_report(test["y"], predictions["test"].argmax(1), target_names=CATEGORIES, digits=3))

In [ ]:
# Matrice de confusion : sur chaque ligne, la vraie catégorie ; en colonne, ce que KORA a prédit.
# Tout ce qui est hors de la diagonale est une erreur.
import matplotlib.pyplot as plt
mc = confusion_matrix(test["y"], predictions["test"].argmax(1), labels=list(range(len(CATEGORIES))))
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(mc, cmap="Oranges")
ax.set_xticks(range(len(CATEGORIES)), CATEGORIES, rotation=35, ha="right")
ax.set_yticks(range(len(CATEGORIES)), CATEGORIES)
for i in range(len(CATEGORIES)):
    for j in range(len(CATEGORIES)):
        ax.text(j, i, mc[i, j], ha="center", va="center", color="white" if mc[i, j] > mc.max() / 2 else "black")
ax.set_xlabel("prédit par KORA"); ax.set_ylabel("vraie catégorie"); ax.set_title(f"{NOM_KORA} — test")
plt.tight_layout(); plt.savefig(f"{BASE}/resultats/matrice_confusion.png", dpi=150); plt.show()

In [ ]:
# Les erreurs : à lire attentivement, c'est là qu'on comprend ce qui manque aux données
erreurs = test.assign(predit=[CATEGORIES[i] for i in predictions["test"].argmax(1)])
erreurs = erreurs[erreurs["predit"] != erreurs["categorie"]]
print(f"{len(erreurs)} erreurs sur {len(test)} messages de test")
erreurs[["categorie", "predit", "texte"]].head(15)

## 6. Comparaison et sauvegarde des résultats

Ce tableau est **la preuve** à mettre dans le pitch : KORA comparé à la référence simple, sur des messages jamais vus.

In [ ]:
import json
tableau = pd.DataFrame(RESULTATS)
display(tableau)
tableau.to_csv(f"{BASE}/resultats/metriques_cerveau.csv", index=False)
with open(f"{BASE}/resultats/metriques_cerveau.json", "w") as f:
    json.dump(RESULTATS, f, ensure_ascii=False, indent=2)
print("💾 resultats/metriques_cerveau.csv")

In [ ]:
# Sauvegarde du modèle KORA sur le Drive (≈ 2 Go)
DOSSIER_KORA = f"{BASE}/modeles/kora-0.01"
if ENTRAINER:
    trainer.save_model(DOSSIER_KORA)
    tokenizer.save_pretrained(DOSSIER_KORA)
    print("💾 Modèle sauvegardé :", DOSSIER_KORA)

### (Optionnel) Publier KORA sur Hugging Face

Publier le modèle prouve qu'il existe et que n'importe qui peut le tester.
1. Crée un *token* d'écriture sur huggingface.co → Settings → Access Tokens.
2. Dans Colab, clique sur la 🔑 (Secrets) à gauche, ajoute `HF_TOKEN` avec ce token, active l'accès au notebook.
3. Mets `PUBLIER = True` et exécute.

Le dépôt est créé en **privé** : tu le rendras public quand l'équipe sera d'accord.

In [ ]:
PUBLIER = False
NOM_DEPOT = "KORA-0.01"

if PUBLIER and ENTRAINER:
    from google.colab import userdata
    from huggingface_hub import login, HfApi
    login(userdata.get("HF_TOKEN"))
    utilisateur = HfApi().whoami()["name"]
    depot = f"{utilisateur}/{NOM_DEPOT}"
    modele.push_to_hub(depot, private=True)
    tokenizer.push_to_hub(depot, private=True)
    print("🚀 Publié (privé) :", f"https://huggingface.co/{depot}")

## 6 bis. Mettre à jour KORA-lite, le modèle de l'application web 📱

L'application en ligne utilise **KORA-lite**, un modèle léger qui tourne **directement dans le téléphone**, sans serveur : aucun message ne quitte l'appareil. Cette cellule le réentraîne avec **toutes vos données**, dont les brouillons fon et les vrais messages si vous les avez, et produit `web/kora_lite.json`.

👉 Ensuite, remplacez le fichier `web/kora_lite.json` du dépôt GitHub par celui-ci (bouton « Add file → Upload files »). Vercel redéploie l'application tout seul.

In [ ]:
import subprocess
fichiers = [f"{BASE}/data/kora_fr.csv"] + [f for f in [f"{BASE}/data/kora_fon_brouillon.csv"] if os.path.exists(f)]
print(subprocess.run([sys.executable, f"{BASE}/lite/entrainer_kora_lite.py", *fichiers], capture_output=True, text=True).stdout[-1500:])

## 7. Essaie toi-même 🎮

Écris un message, KORA te répond. C'est exactement ce que fera la démo.

In [ ]:
def kora(message):
    p = probas_kora([normaliser(message)])[0]
    r = expliquer(message, {c: float(p[i]) for i, c in enumerate(CATEGORIES)})
    print(f"\n📩 {message}\n{r['titre']}  (risque {r['risque']:.0%}) — {r['type']}")
    for raison in r["raisons"]:
        print("   •", raison)
    print("   💡", r["conseil"])

kora("Bonjour je suis agent MTN, votre compte va être bloqué, envoyez moi le code que vous allez recevoir")
kora("Vous avez reçu 10 000 FCFA de AGBO Koffi. Nouveau solde : 12 500 FCFA. ID : 8F3A2C91D0")
kora("Felicitation vous avez gagne 300.000F au tirage Moov, envoyez 5000F de frais pour retirer")
kora("Maman j'ai envoyé l'argent du marché, regarde ton solde")

## ✅ Récapitulatif

- `modeles/baseline.joblib` : la référence simple (roue de secours de la démo)
- `modeles/kora-0.01/` : **le modèle KORA** (si entraîné sur GPU)
- `resultats/metriques_cerveau.csv`, `matrice_confusion.png`, `courbes_apprentissage.png` : **les preuves pour le pitch**

### 🎤 Ce que tu dis au jury
> « KORA-0.01 est un modèle **AfroXLMR-large**, pré-entraîné sur 76 langues africaines dont le fon, que nous avons **affiné sur notre propre jeu de données** d'arnaques Mobile Money béninoises. Sur des modèles d'arnaques **jamais vus pendant l'entraînement**, il détecte **X %** des arnaques avec **Y %** de fausses alertes, contre **Z %** pour une méthode classique. »

(Remplace X, Y, Z par les chiffres du tableau ci-dessus. Pour la baseline, donne le chiffre de la **validation croisée** avec son écart, par exemple « 85 % ± 10 % ».)

### ⚠️ Limites à annoncer honnêtement (le jury les demande)
- Les données sont **synthétiques** : plusieurs formulations d'un même type d'arnaque se ressemblent, donc les scores sur le test sont **optimistes**. Le vrai juge, ce sont les **vrais messages** récoltés.
- La validation croisée n'est faite que pour la baseline (rapide). Le gros modèle est mesuré sur un seul découpage, faute de temps de GPU.
- Le fon écrit vient de traductions automatiques, à confirmer avec des traductions humaines.

➡️ **Suite : Notebook 3 — l'oreille fon.**